In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
import textwrap
from pydantic import BaseModel, field_validator
from langchain.messages import SystemMessage

chat_model = ChatOpenAI(model="gpt-4o-mini")


def indent(content: str, prefix: str) -> str:
    return textwrap.indent(content, prefix, lambda line: True).strip()


class StructuredResponse[T](BaseModel):
    reasoning: str
    errors: list[str]
    result: T


class Example(BaseModel):
    user_query: str
    assistance_response: str

    def __str__(self) -> str:
        return f"""<user_query>
{self.user_query}
</user_query>
<assistance_response>
{self.assistance_response}
</assistance_response>
"""


class MultipleExamples:
    def __init__(self, examples: list[Example]):
        self._examples = examples

    def __str__(self) -> str:
        content = ""
        for index, example in enumerate(self._examples):
            content += f"## Example {index + 1}"
            content += str(example)

        return f"""# Examples
{content}
"""


class MultipleInstructions:
    def __init__(self, instructions: list[str]):
        self._instructions = instructions

    def __str__(self) -> str:
        content = ""
        for index, instruction in enumerate(self._instructions):
            content += f"{index + 1}. {indent(instruction, ". ")}\n"

        return f"""# Instructions
{content}
"""


def ai_assert(question: str) -> None:
    pass


class Identity(BaseModel):
    content: str | None = None

    @field_validator("content", mode="after")
    def validate_content(self) -> str | None:
        ai_assert(question=f"Does this content answer for the question who? {self.content}")
        ai_assert(question=f"Noi dung nay the hien vai tro cua openAI trong cuoc hoi thoai? {self.content}")

        return self.content

    def __str__(self) -> str:
        if not self.content:
            return ""
        return f"""# Identity
{self.content}
"""


class IdentityCreator:
    def __init__(self,  content: StructuredSystemContent):
        self._content = content

    def create(self) -> str:
        return f"Detect identity for the content\n > {str(self._content)}"
        ...


class MultipleExamplesCreator:
    def __init__(self, content: StructuredSystemContent):
        self._content = content

    def create(self) -> list[Example]:
        ...


class MultipleInstructionsCreator:
    def __init__(self, content: StructuredSystemContent):
        self._content = content

    def create(self) -> list[str]:
        ...


class StructuredSystemContent(BaseModel):
    identity: str | None = None
    instructions: list[str] = []
    examples: list[Example] = []

    def __str__(self) -> str:
        return f"""{str(Identity(content=self.identity))}
{str(MultipleInstructions(self.instructions))}
{str(MultipleExamples(self.examples))}
"""

    @field_validator("identity", mode="after")
    def init_identity(self) -> str:
        if self.identity:
            return self.identity
        return IdentityCreator(self).create()

    @field_validator("examples", mode="after")
    def init_examples(self) -> list[Example]:
        if self.examples:
            return self.examples
        return MultipleExamplesCreator(self).create()

    @field_validator("instructions", mode="after")
    def init_instructions(self) -> list[str]:
        if self.instructions:
            return self.instructions
        return MultipleInstructionsCreator(self).create()


def main():
    output_parser = PydanticOutputParser(pydantic_object=StructuredResponse)

    system_message = SystemMessage(content=str(StructuredSystemContent(
        identity="",
        examples=[],
        instructions=[
            # ...
            output_parser.get_format_instructions()
        ],
    )))

    response = chat_model.invoke([
        system_message,
        HumanMessage(content="")
    ])
    print(response)


if __name__ == "__main__":
    main()
